# jev-local Colabランチャー

このノートブックはリポジトリをcloneして依存をインストールするだけ。
実際のロジックは全てリポジトリ側のモジュールにある。
生成物・チェックポイントはGoogle DriveかHF Hubに保存すること。

In [ ]:
!git clone https://github.com/fukayatti/jev-japanese-judgment.git
%cd jev-japanese-judgment
!pip install -q -r requirements.txt
!pip install -q vllm
# vllmがtorchを別のCUDAビルドに上げてしまい、Colab既存のtorchaudio/torchvisionと
# CUDAバージョンが食い違ってRuntimeErrorになることがある。このプロジェクトは
# 音声・画像処理を使わないので、素直に外してしまうのが一番安定する。
# torchaoもColabに古いバージョン(0.10.0)がプリインストールされており、
# peftが要求する0.16.0以上と食い違ってImportErrorになる。これも未使用なので外す。
!pip uninstall -y -q torchaudio torchvision torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/jev-japanese-judgment-checkpoints'

In [ ]:
# Colabのユーザーシークレット(左メニューの鍵アイコン)に HF_TOKEN を登録しておくこと
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
import subprocess
from pathlib import Path

from data.convert import chabsa, jcommonsenseqa, jsnli

# JCommonsenseQA / chABSA は HF Hub (parquet) から直接ロードできる
jcqa_examples = jcommonsenseqa.convert('train')
chabsa_examples = chabsa.convert('train')

# JSNLIは配布形式がzipなのでダウンロード・展開してからパスを渡す
jsnli_dir = Path('data/raw/jsnli')
jsnli_dir.mkdir(parents=True, exist_ok=True)
if not (jsnli_dir / 'jsnli_1.1' / 'train_w_filtering.tsv').exists():
    subprocess.run(['curl', '-sL', '-o', str(jsnli_dir / 'jsnli.zip'),
                     'https://nlp.ist.i.kyoto-u.ac.jp/nl-resource/JSNLI/jsnli_1.1.zip'], check=True)
    subprocess.run(['unzip', '-o', '-q', str(jsnli_dir / 'jsnli.zip'), '-d', str(jsnli_dir)], check=True)

jsnli_examples = jsnli.convert(jsnli_dir / 'jsnli_1.1' / 'train_w_filtering.tsv')

all_examples = jcqa_examples + chabsa_examples + jsnli_examples
print(f'jcqa={len(jcqa_examples)} chabsa={len(chabsa_examples)} jsnli={len(jsnli_examples)} total={len(all_examples)}')

In [ ]:
from data.augment.generate import build_prompts, run_batch_generation

# vLLM本体はsubprocess (data/augment/vllm_worker.py) 側で実行する。
# ノートブックのカーネル内で直接vllm.LLM(...)を呼ぶと、CUDA初期化済みの
# プロセスからspawnしようとしてデッドロックすることがあるため。
#
# エンジン初期化だけで約9分かかる(T4でのTritonカーネルJITコンパイル)ので、
# 複数回に分けず一気に実行すること。ただしall_examples全件(約19,000件 x 3
# プロンプト=57,000件)をT4で回すと現実的な時間を超えるため、まずは一部だけ
# 拡張する。件数を増やしたい場合はこの3000という数字を変えること。
AUGMENT_TARGET_SIZE = 3000
jobs = build_prompts(all_examples[:AUGMENT_TARGET_SIZE])
results = run_batch_generation(jobs)
for r in results[:5]:
    print(r['kind'], '->', r['output'])

In [ ]:
from data.augment.merge import merge_augmentations
from data.postprocess.shuffle_candidates import shuffle_all

augmented_examples = merge_augmentations(all_examples, results)
print(f'augmented examples created: {len(augmented_examples)}')

# 公開用データセット = 元データ + 拡張データ。候補の順序をシャッフルし
# labelインデックスを再計算してから、以降の学習・公開で使う。
final_examples = shuffle_all(all_examples + augmented_examples)
print(f'final_examples total: {len(final_examples)}')

In [ ]:
from train import main as train_main

model = train_main(final_examples)

In [ ]:
from scripts.push_to_hub import push

# HF_TOKENは前段のセルで環境変数にセット済み。ここではrepo_idのみ指定する
push(final_examples, repo_id="fukayatti/jev-japanese-judgment", private=False)